In [0]:
# ================================================================
# FEATURE CONFIGURATION - SINGLE SOURCE OF TRUTH
# ================================================================
# This should match the configuration in the training notebook.
# Ideally, this would be in a shared module, but for now we duplicate it.

DISCRETE_FEATURES = [
    {'col': 'Driver_idx', 'emb_dim': 8, 'zero_based': False},
    {'col': 'Team_idx', 'emb_dim': 8, 'zero_based': False},
    {'col': 'Compound', 'emb_dim': 4, 'zero_based': True},  # Tyre compound: convert 1..n to 0..n-1
    {'col': 'Circuit_Name', 'emb_dim': 12, 'zero_based': False}  # Circuit identity to learn baseline pace
]

def get_discrete_col(role):
    """Get discrete feature column name by role (driver, team, tyre, or circuit)."""
    role_map = {
        'driver': 'Driver_idx',
        'team': 'Team_idx',
        'tyre': 'Compound',
        'circuit': 'Circuit_Name'
    }
    return role_map.get(role)

def should_normalize_to_zero_based(col_name):
    """Check if column should be converted to 0-based indexing."""
    for f in DISCRETE_FEATURES:
        if f['col'] == col_name:
            return f['zero_based']
    return False

In [0]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

In [0]:
def assign_stints(df, car_col='Driver_idx', session_col='Circuit_Name', tyre_life_col='TyreLife'):
    df = df.sort_values([session_col, car_col, 'LapNumber'])  # no reset_index here
    df = df.reset_index().rename(columns={'index': 'orig_index'})  # preserve original index as a column
    df['NewStint'] = df[tyre_life_col] == 0
    df['StintID'] = df.groupby([session_col, car_col])['NewStint'].cumsum()
    return df

In [0]:
class StintDataset(Dataset):
    def __init__(self, df, cont_features, target='LapTime_sec',
                 driver_col=None, team_col=None, tyre_col=None, circuit_col=None):
        """
        Args:
            df: DataFrame with stint data
            cont_features: List of continuous feature column names
            target: Target column name (default: 'LapTime_sec')
            driver_col: Override driver column (default: uses DISCRETE_FEATURES config)
            team_col: Override team column (default: uses DISCRETE_FEATURES config)
            tyre_col: Override tyre column (default: uses DISCRETE_FEATURES config)
            circuit_col: Override circuit column (default: uses DISCRETE_FEATURES config)
        """
        # Use config defaults if not specified
        if driver_col is None:
            driver_col = get_discrete_col('driver')
        if team_col is None:
            team_col = get_discrete_col('team')
        if tyre_col is None:
            tyre_col = get_discrete_col('tyre')
        if circuit_col is None:
            circuit_col = get_discrete_col('circuit')
        
        # Work on a local copy so we don't mutate the caller DataFrame
        df_local = df.copy()
        
        # Convert Circuit_Name strings to indices if present
        if circuit_col and circuit_col in df_local.columns:
            if df_local[circuit_col].dtype == 'object':  # String column
                unique_circuits = df_local[circuit_col].unique()
                circuit_to_idx = {name: idx for idx, name in enumerate(sorted(unique_circuits))}
                df_local[f'{circuit_col}_idx'] = df_local[circuit_col].map(circuit_to_idx)
                circuit_col = f'{circuit_col}_idx'  # Use the new indexed column
        
        # Normalize discrete features to 0-based if configured
        if tyre_col in df_local.columns and should_normalize_to_zero_based(tyre_col):
            df_local[tyre_col] = df_local[tyre_col].astype(int) - 1
            # Validate indices
            if df_local[tyre_col].min() < 0:
                raise ValueError(f"Found {tyre_col} index < 0 after normalization")

        self.stints = []
        for (session, car, stint_id), stint_df in df_local.groupby(['Circuit_Name', 'Driver_idx', 'StintID']):
            stint_dict = {
                'cont_feats': torch.tensor(stint_df[cont_features].values, dtype=torch.float),
                'driver_idx': int(stint_df[driver_col].iloc[0]),
                'team_idx': int(stint_df[team_col].iloc[0]),
                'tyre_idx': torch.tensor(stint_df[tyre_col].values, dtype=torch.long) if tyre_col in stint_df.columns else torch.tensor([], dtype=torch.long),
                'lap_time': torch.tensor(stint_df[target].values, dtype=torch.float),
                'indices': stint_df['orig_index'].tolist()
            }
            
            # Add circuit_idx if available
            if circuit_col and circuit_col in stint_df.columns:
                stint_dict['circuit_idx'] = int(stint_df[circuit_col].iloc[0])
            
            self.stints.append(stint_dict)

    def __len__(self):
        return len(self.stints)

    def __getitem__(self, idx):
        return self.stints[idx]

In [0]:
# ----------------------
# Collate function
# ----------------------
def collate_fn(batch):
    cont_feats_list = [b['cont_feats'] for b in batch]
    driver_idx = torch.tensor([b['driver_idx'] for b in batch], dtype=torch.long)
    team_idx = torch.tensor([b['team_idx'] for b in batch], dtype=torch.long)
    tyre_idx_list = [b['tyre_idx'] for b in batch]
    lap_time_list = [b['lap_time'] for b in batch]
    
    # Extract circuit_idx if present (all stints in a batch should have it or none)
    circuit_idx = None
    if 'circuit_idx' in batch[0]:
        circuit_idx = torch.tensor([b['circuit_idx'] for b in batch], dtype=torch.long)

    cont_feats_padded = pad_sequence(cont_feats_list, batch_first=True, padding_value=0.0)
    tyre_idx_padded = pad_sequence(tyre_idx_list, batch_first=True, padding_value=0)
    lap_time_padded = pad_sequence(lap_time_list, batch_first=True, padding_value=0.0)

    seq_lengths = torch.tensor([len(b['lap_time']) for b in batch])
    max_len = cont_feats_padded.size(1)
    mask = torch.arange(max_len).expand(len(batch), max_len) >= seq_lengths.unsqueeze(1)

    return cont_feats_padded, driver_idx, team_idx, tyre_idx_padded, circuit_idx, lap_time_padded, mask